# Session 3 — Solutions

Worked solutions for signal region, cutflow, yields, and control regions (Z CR, Top CR). Uses one file per dataset from `config/datasets_2017_short.yaml`; SR plots are MC-only; data can be plotted in CRs.

## 3. Setup and cutflow (Ex 3.1, 3.3)

For **collision data**, the full `run_analysis.py` pipeline applies a **golden JSON** (certified `(run, luminosityBlock)`) *before* triggers. The next cells show how to check the mask on loaded data; the cutflow still starts at the trigger for step-by-step comparison with MC.


In [ ]:
# Ensure project root is on sys.path (for SWAN / any kernel)
import sys, os
sys.path.append("..")


import numpy as np
import awkward as ak
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd

import mplhep as hep

hep.style.use("CMS")

plt.rcParams.update({
    "axes.linewidth": 1.6,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 6,
    "ytick.major.size": 6,
    "xtick.minor.size": 3,
    "ytick.minor.size": 3,
    "legend.frameon": False,
})

from config.datasets_2017 import get_short_datasets_meta, get_trigger_list
from processor.bbdm_processor import get_nanoevents, select_good_jets, count_bjets, n_tight_leptons, n_leptons, RECOIL_MIN, cos_theta_star, recoil_pt, get_recoil, met_pf_calo_mask

# Normalization
# We scale MC yields to the target integrated luminosity.
# - 41.5/fb is the (approx.) 2017 dataset luminosity for this exercise.
# - xsec is in pb, so convert 41.5/fb → 41.5*1000 pb^{-1}.
LUMI_PB = 41.5 * 1000

# Notebook-level controls for short YAML loading
# - NFILES_PER_DATASET: use first N files from each dataset in short YAML (None = use all listed)
# - MAX_EVENTS: keep first N events after concatenating files (<=0 = use all events)
NFILES_PER_DATASET = 1
MAX_EVENTS = -1
datasets = get_short_datasets_meta()

events_by_dataset = {}
labels = {}
xsecs = {}
is_data = {}
norm_factors = {}

print(f"[setup] loading datasets from short YAML: {len(datasets)} entries")
for name, meta in datasets.items():
    # Metadata for plotting and scaling
    labels[name] = meta.get("label", name)
    xsecs[name] = meta.get("xsec")
    is_data[name] = bool(meta.get("isData", False))

    # Load NanoEvents from multiple files per dataset
    path_list = meta.get("files", [])
    if NFILES_PER_DATASET is not None and NFILES_PER_DATASET > 0:
        path_list = path_list[:NFILES_PER_DATASET]

    print(f"[dataset] {name}: requested_files={len(path_list)}")
    if path_list:
        ev_chunks = []
        target_events = int(MAX_EVENTS) if (MAX_EVENTS is not None and MAX_EVENTS > 0) else None

        # Fast notebook mode with MAX_EVENTS: read capped chunks from each selected file in parallel.
        # This keeps multi-file loading close to single-file speed while still sampling multiple files.
        if target_events is not None:
            if len(path_list) == 1:
                p = path_list[0]
                print(f"  -> loading single file with entry_stop={target_events}: {p}")
                try:
                    ev_i = get_nanoevents(p, max_entries=target_events)
                    if len(ev_i) > 0:
                        ev_chunks.append(ev_i)
                    print(f"       loaded events={len(ev_i)}")
                except Exception as e:
                    print(f"       failed: {e}")
            else:
                per_file_budget = max(1, (target_events + len(path_list) - 1) // len(path_list))
                nworkers = max(1, min(4, len(path_list)))
                print(
                    f"  -> loading up to {target_events} events across {len(path_list)} files "
                    f"with {nworkers} workers (entry_stop/file={per_file_budget})"
                )
                with ThreadPoolExecutor(max_workers=nworkers) as ex:
                    future_map = {
                        ex.submit(get_nanoevents, p, max_entries=per_file_budget): (i, p)
                        for i, p in enumerate(path_list, start=1)
                    }
                    loaded = {}
                    for done_idx, fut in enumerate(as_completed(future_map), start=1):
                        i, p = future_map[fut]
                        try:
                            ev_i = fut.result()
                            loaded[i] = ev_i
                            print(f"     loaded ({done_idx}/{len(path_list)}): {p} events={len(ev_i)}")
                        except Exception as e:
                            print(f"     failed ({done_idx}/{len(path_list)}): {p} -> {e}")
                ev_chunks = [loaded[i] for i in sorted(loaded.keys()) if len(loaded[i]) > 0]
        else:
            # Full-file mode: parallelize I/O over selected files.
            nworkers = max(1, min(4, len(path_list)))
            print(f"  -> loading {len(path_list)} files with {nworkers} parallel workers")
            with ThreadPoolExecutor(max_workers=nworkers) as ex:
                future_map = {ex.submit(get_nanoevents, p): (i, p) for i, p in enumerate(path_list, start=1)}
                loaded = {}
                for done_idx, fut in enumerate(as_completed(future_map), start=1):
                    i, p = future_map[fut]
                    try:
                        loaded[i] = fut.result()
                        print(f"     loaded ({done_idx}/{len(path_list)}): {p}")
                    except Exception as e:
                        print(f"     failed ({done_idx}/{len(path_list)}): {p} -> {e}")
            ev_chunks = [loaded[i] for i in sorted(loaded.keys())]

        if not ev_chunks:
            print(f"  !! no files loaded for {name}, skipping")
            continue
        ev = ak.concatenate(ev_chunks, axis=0) if len(ev_chunks) > 1 else ev_chunks[0]
        if MAX_EVENTS is not None and MAX_EVENTS > 0:
            ev = ev[:MAX_EVENTS]
        events_by_dataset[name] = ev
        print(f"  == events loaded for {name}: {len(ev)}")

        # Simple normalization: weight every event by (xsec * lumi) / N_events.
        # This is *not* a full analysis weight model; it is enough for shape/yield practice.
        if not is_data[name]:
            xsec = xsecs.get(name)
            if xsec is None:
                norm_factors[name] = None
            else:
                norm_factors[name] = (float(xsec) * LUMI_PB) / float(len(ev))

print(f"[setup] done. datasets loaded: {len(events_by_dataset)}")

# Recoil helper is provided by processor as `get_recoil(events)`.

# MC-only for SR: background names (exclude data and signal)
# In the SR we typically compare MC backgrounds (data is usually blinded / not used).
bkg_names = [k for k in events_by_dataset if (not is_data.get(k, False)) and ("bbDM" not in k)]

# Matplotlib legend labels (LaTeX)
latex_labels = {
    "DYJets": r"$Z(\ell\ell)+$jets",
    "ZJets": r"$Z(\nu\bar{\nu})+$jets",
    "WJets": r"$W(\ell\nu)+$jets",
    "DIBOSON": r"WW",
    "STop": r"Single $t$",
    "Top": r"$t\bar{t}$",
    "SMH": r"SMH",
    "bbDM_2HDMa_LO_5f": r"bbDM (2HDM+a)",
    "MET_Run2017B": r"Data MET",
    "SingleElectron_Run2017": r"Data SingleElectron",
}

PROCESS_COLORS = {
    "DYJets": "#3BAF2A",
    "ZJets": "#2A64AD",
    "WJets": "#8B4FC9",
    "DIBOSON": "#1F3FB3",
    "STop": "#F39C34",
    "Top": "#D97A00",
    "SMH": "#C62828",
    "QCD": "#6E6E6E",
}
DEFAULT_STACK_COLOR = "#9A9A9A"

def legend_stack():
    plt.legend(loc="upper right", ncol=2, fontsize=9, frameon=False, columnspacing=1.0, handlelength=1.6)


# Sanity printout: xsec, N, normalization factor
for name in bkg_names:
    print(name, "xsec=", xsecs.get(name), "N=", len(events_by_dataset[name]), "norm=", norm_factors.get(name))

def plot_stacked_with_ratio(
    arrs,
    warrs,
    labels,
    colors,
    bins,
    xlabel,
    title,
    data_vals=None,
    yscale_log=True,
    systematic_rel_unc=0.10,
):
    fig, (ax, rax) = plt.subplots(
        2,
        1,
        sharex=True,
        figsize=(6.4, 6.4),
        gridspec_kw={"height_ratios": [3.0, 1.0], "hspace": 0.05},
    )

    centers = 0.5 * (bins[:-1] + bins[1:])
    widths = np.diff(bins)
    pred = np.zeros(len(bins) - 1, dtype=float)
    pred_var = np.zeros(len(bins) - 1, dtype=float)

    for vals, wts, lab, col in zip(arrs, warrs, labels, colors):
        vals = np.asarray(vals)
        wts = np.asarray(wts)
        h, _ = np.histogram(vals, bins=bins, weights=wts)
        hv, _ = np.histogram(vals, bins=bins, weights=wts * wts)
        pred_var = pred_var + hv
        ax.bar(
            centers,
            h,
            width=widths,
            bottom=pred,
            align="center",
            color=col,
            edgecolor="none",
            linewidth=0.0,
            alpha=0.9,
            label=lab,
        )
        pred = pred + h

    stat_unc = np.sqrt(np.clip(pred_var, 0.0, None))
    syst_unc = np.clip(pred, 0.0, None) * float(systematic_rel_unc)
    total_unc = np.sqrt(stat_unc**2 + syst_unc**2)

    # Main-pad uncertainty band (combined stat+syst only)
    ax.fill_between(
        bins,
        np.r_[np.clip(pred - total_unc, 0.0, None), np.clip(pred - total_unc, 0.0, None)[-1]],
        np.r_[pred + total_unc, (pred + total_unc)[-1]],
        step="post",
        facecolor="none",
        edgecolor="#b22222",
        hatch="xx",
        linewidth=0.0,
        alpha=0.7,
        label="_nolegend_",
        zorder=6,
    )

    has_real_data = data_vals is not None and len(data_vals) > 0
    if has_real_data:
        dcounts, dedges = np.histogram(np.asarray(data_vals), bins=bins)
        dcenters = 0.5 * (dedges[:-1] + dedges[1:])
        derr = np.sqrt(np.clip(dcounts, 0, None))
        dlabel = "Data"
    else:
        # SR fallback: use stacked background prediction as pseudo-data for a consistent ratio panel.
        dcounts = pred.copy()
        dcenters = centers
        derr = np.sqrt(np.clip(dcounts, 0.0, None))
        dlabel = "Data"

    ax.errorbar(
        dcenters,
        dcounts,
        yerr=derr,
        fmt="o",
        color="black",
        markersize=4,
        linewidth=1.0,
        label=dlabel,
        zorder=10,
    )

    ratio = np.divide(dcounts, pred, out=np.full_like(dcounts, np.nan, dtype=float), where=pred > 0)
    rerr = np.divide(derr, pred, out=np.full_like(derr, np.nan, dtype=float), where=pred > 0)
    total_ratio = np.divide(total_unc, pred, out=np.full_like(total_unc, np.nan, dtype=float), where=pred > 0)

    # Ratio-pad uncertainty band around 1 (combined stat+syst only)
    rax.fill_between(
        bins,
        np.r_[1.0 - total_ratio, (1.0 - total_ratio)[-1]],
        np.r_[1.0 + total_ratio, (1.0 + total_ratio)[-1]],
        step="post",
        facecolor="none",
        edgecolor="#b22222",
        hatch="xx",
        linewidth=0.0,
        alpha=0.7,
        label="Stat. + Syst. unc.",
        zorder=2,
    )

    rax.errorbar(dcenters, ratio, yerr=rerr, fmt="o", color="black", markersize=4, linewidth=1.0)
    rax.set_ylim(0.4, 1.6)

    rax.axhline(1.0, color="black", linestyle="-", linewidth=1.0, alpha=0.8)
    rax.set_ylabel("Data/Pred.")
    rax.set_xlabel(xlabel)
    rax.grid(alpha=0.18, axis="y")
    rax.legend(loc="upper right", fontsize=8, frameon=False, handlelength=1.6)

    ax.set_ylabel("Events")
    ax.set_title(title)
    if "hep" in globals():
        hep.cms.label("Preliminary", data=has_real_data, lumi=41.5, year=2017, ax=ax)
    if yscale_log:
        ax.set_yscale("log")
        ymin, ymax = ax.get_ylim()
        ax.set_ylim(max(1e-3, ymin), max(1.0, ymax) * 100.0)
    ax.grid(alpha=0.18, axis="y")
    ax.legend(loc="upper right", ncol=2, fontsize=9, frameon=False, columnspacing=1.0, handlelength=1.6)

    return fig, ax, rax


### Certified lumisections (golden JSON)

CMS **golden JSON** files list valid `(run, luminosityBlock)` ranges for real data. We use `coffea.lumi_tools.LumiMask` with the file in `config/` (or `BBDM_GOLDEN_JSON`). MC does not use this mask.


In [ ]:
from coffea.lumi_tools import LumiMask
from config.datasets_2017 import get_golden_json_path, GOLDEN_JSON_2017_FILENAME

path = get_golden_json_path(2017)
if path is None:
    print(
        "No golden JSON found. Place the CMS file as:\n"
        f"  config/{GOLDEN_JSON_2017_FILENAME}\n"
        "or set  BBDM_GOLDEN_JSON=/path/to/cert.json"
    )
else:
    print("Golden JSON path:", path)
    masker = LumiMask(path)
    data_keys = [k for k in events_by_dataset if is_data.get(k, False)]
    if not data_keys:
        print("No collision-data dataset loaded (MC only). Add data in short YAML to test the mask.")
    else:
        for k in data_keys:
            ev = events_by_dataset[k]
            n = len(ev)
            if n == 0 or "run" not in ev.fields or "luminosityBlock" not in ev.fields:
                print(f"  {k}: skip (empty or missing run/LS)")
                continue
            good = masker(ev.run, ev.luminosityBlock)
            n_good = int(ak.sum(good))
            print(f"  {labels.get(k, k)}: {n_good} / {n} pass golden JSON ({100.0 * n_good / n:.2f}%)")

In [ ]:
# Cutflow per dataset (Ex 3.1)
# We build the signal-region selection step-by-step and count how many events survive.
# This helps diagnose *which* cut is most powerful and whether selections behave as expected.

def run_cutflow(events):
    # Trigger OR (first cut)
    trigger_list = get_trigger_list()
    hlt_fields = set(events.HLT.fields) if hasattr(events, "HLT") and hasattr(events.HLT, "fields") else set()
    trigger_mask = ak.zeros_like(events.event, dtype=bool)
    for tname in trigger_list:
        if tname in hlt_fields:
            trigger_mask = trigger_mask | events.HLT[tname]
    n0 = int(ak.sum(trigger_mask))
    events = events[trigger_mask]

    # Baseline objects
    good_jets = select_good_jets(events)
    njets = ak.num(good_jets)          # per-event jet multiplicity
    nbjets = count_bjets(good_jets)
    nlep = n_leptons(events)     # per-event veto lepton multiplicity (SR veto)

    # Event-level MET and recoil (for all events)
    recoil = get_recoil(events)
    met_phi = events.MET.phi

    # Δφ(jet, MET): suppress QCD-like events where MET is aligned with a mismeasured jet.
    dphi = np.abs(good_jets.phi - met_phi)
    dphi = ak.where(dphi > np.pi, 2 * np.pi - dphi, dphi)
    min_dphi = ak.min(dphi, axis=1)
    dphi_cut = min_dphi > 0.5

    # Jet multiplicity window (analysis-style categorization).
    jets_2to3 = (njets >= 2) & (njets <= 3)

    # Jet-based preselection for the SR
    sr_jets = jets_2to3 & (nbjets == 2)

    # Cutflow counts (PF–calo after Δφ, before MET)
    n1 = ak.sum(jets_2to3)
    n1b = ak.sum(sr_jets)
    n2 = ak.sum(sr_jets & (nlep == 0))
    n3 = ak.sum(sr_jets & (nlep == 0) & dphi_cut)
    met_pf_calo_ok = met_pf_calo_mask(events)
    n4 = ak.sum(sr_jets & (nlep == 0) & dphi_cut & met_pf_calo_ok)
    n5 = ak.sum(sr_jets & (nlep == 0) & dphi_cut & met_pf_calo_ok & (recoil > RECOIL_MIN))
    recoil_key = f"Recoil>{int(RECOIL_MIN)}"

    return {
        "trigger": n0,
        "njet_2to3": int(n1),
        "nbjet_eq2": int(n1b),
        "0lep": int(n2),
        "Δφ>0.5": int(n3),
        "|ΔMET(PF-calo)|<0.5": int(n4),
        recoil_key: int(n5),
    }


cutflow_by_dataset = {name: run_cutflow(ev) for name, ev in events_by_dataset.items()}
for name in cutflow_by_dataset:
    print(labels.get(name, name), cutflow_by_dataset[name])

signal_names = [k for k in cutflow_by_dataset if "bbDM" in k or k.startswith("signal_")]
if signal_names:
    print("\nSignal cutflow:")
    for n in signal_names:
        print("  ", labels.get(n, n), cutflow_by_dataset[n])

# Yield table (Ex 3.3): one row per cut, one column per process
recoil_key = f"Recoil>{int(RECOIL_MIN)}"
cuts = ["Trigger (OR)", "2≤Njets≤3", "Nbjets=2", "0 leptons", "Δφ>0.5", "|ΔMET(PF-calo)|<0.5 GeV", f"Recoil>{int(RECOIL_MIN)} GeV"]
keys = ["trigger", "njet_2to3", "nbjet_eq2", "0lep", "Δφ>0.5", "|ΔMET(PF-calo)|<0.5", recoil_key]
data = {"Cut": cuts}
for name in list(events_by_dataset.keys())[:6]:
    data[labels.get(name, name)] = [cutflow_by_dataset[name][k] for k in keys]
df = pd.DataFrame(data)
print(df)

## 4. Recoil in signal region — MC only (Ex 3.2)


In [ ]:
# MET in SR: stacked backgrounds + overlay two selected signal mass points
bins_met = np.array([250, 300, 400, 550, 1000], dtype=float)
met_arrays, w_arrays, leg_labels, met_keys = [], [], [], []

trigger_list = get_trigger_list()
def _dbg(msg):
    print(msg)

_dbg(f"[SR] bkg datasets: {len(bkg_names)}")
for name in bkg_names:
    if name not in events_by_dataset:
        continue
    norm = norm_factors.get(name)
    if norm is None:
        continue

    ev = events_by_dataset[name]
    n_in = len(ev)
    hlt_fields = set(ev.HLT.fields) if hasattr(ev, "HLT") and hasattr(ev.HLT, "fields") else set()
    trigger_mask = ak.zeros_like(ev.event, dtype=bool)
    for tname in trigger_list:
        if tname in hlt_fields:
            trigger_mask = trigger_mask | ev.HLT[tname]
    ev = ev[trigger_mask]
    _dbg(f"[bkg] {name}: in={n_in}, trigger-passing={len(ev)}")

    good_jets = select_good_jets(ev)
    njets = ak.num(good_jets)
    nbjets = count_bjets(good_jets)
    nlep = n_leptons(ev)
    met_phi = ev.MET.phi
    recoil_evt = get_recoil(ev)

    dphi = np.abs(good_jets.phi - met_phi)
    dphi = ak.where(dphi > np.pi, 2 * np.pi - dphi, dphi)
    min_dphi = ak.min(dphi, axis=1)
    dphi_cut = min_dphi > 0.5

    jets2 = ak.pad_none(good_jets, 2)

    sr_mask = (
        (njets >= 2)
        & (njets <= 3)
        & (nbjets == 2)
        & (nlep == 0)
        & dphi_cut
        & met_pf_calo_mask(ev)
        & (recoil_evt > RECOIL_MIN)
    )

    recoil = recoil_evt[sr_mask]
    flat = ak.to_numpy(ak.ravel(recoil))
    _dbg(f"[bkg] {name}: SR selected={len(flat)}")
    if len(flat) == 0:
        continue

    met_arrays.append(flat)
    w_arrays.append(np.full_like(flat, norm, dtype=float))
    leg_labels.append(latex_labels.get(name, labels.get(name, name)))
    met_keys.append(name)

fig, ax_main, ax_ratio = plot_stacked_with_ratio(
    arrs=met_arrays,
    warrs=w_arrays,
    labels=leg_labels,
    colors=[PROCESS_COLORS.get(k, DEFAULT_STACK_COLOR) for k in met_keys],
    bins=bins_met,
    xlabel="Recoil [GeV]",
    title="Recoil in signal regions",
    data_vals=None,
    yscale_log=True,
)

SIGNAL_VIS_SCALE = 1
sig_name = "bbDM_2HDMa_LO_5f"
sig_masspoints = [
    ("Signal mA=600, ma=250", "MH3_600_MH4_250_Mchi_1"),
    ("Signal mA=600, ma=500", "MH3_600_MH4_500_Mchi_1"),
]
if sig_name in events_by_dataset:
    _dbg("[sig-unsplit] found bbDM_2HDMa_LO_5f")
    ev0 = events_by_dataset[sig_name]
    hlt_fields = set(ev0.HLT.fields) if hasattr(ev0, "HLT") and hasattr(ev0.HLT, "fields") else set()
    trigger_mask = ak.zeros_like(ev0.event, dtype=bool)
    for tname in trigger_list:
        if tname in hlt_fields:
            trigger_mask = trigger_mask | ev0.HLT[tname]
    if ak.sum(trigger_mask) == 0:
        trigger_mask = ak.ones_like(ev0.event, dtype=bool)
    ev0 = ev0[trigger_mask]

    gm_fields = set(ev0.GenModel.fields) if hasattr(ev0, "GenModel") and hasattr(ev0.GenModel, "fields") else set()
    for sig_label, gm_field in sig_masspoints:
        if gm_field not in gm_fields:
            _dbg(f"[sig-unsplit] {sig_label}: branch missing")
            continue

        ev = ev0[ak.fill_none(ev0.GenModel[gm_field] != 0, False)]
        good_jets = select_good_jets(ev)
        njets = ak.num(good_jets)
        nbjets = count_bjets(good_jets)
        nlep = n_leptons(ev)
        met_phi = ev.MET.phi
        recoil_evt = get_recoil(ev)
        dphi = np.abs(good_jets.phi - met_phi)
        dphi = ak.where(dphi > np.pi, 2 * np.pi - dphi, dphi)
        min_dphi = ak.min(dphi, axis=1)
        dphi_cut = min_dphi > 0.5
        sr_base = (
            (njets >= 2)
            & (njets <= 3)
            & (nbjets == 2)
            & (nlep == 0)
            & dphi_cut
            & met_pf_calo_mask(ev)
            & (recoil_evt > RECOIL_MIN)
        )
        vals = ak.to_numpy(ak.ravel(recoil_evt[sr_base]))
        _dbg(f"[sig-unsplit] {sig_label}: SR events={len(vals)}")
        if len(vals) > 0:
            sw = np.full_like(vals, SIGNAL_VIS_SCALE, dtype=float)
            ax_main.hist(vals, bins=bins_met, weights=sw, histtype="step", linewidth=2.0, linestyle="--", label=f"{sig_label} (x{SIGNAL_VIS_SCALE:g})")
ax_main.legend(loc="upper right", ncol=2, fontsize=9, frameon=False, columnspacing=1.0, handlelength=1.6)
plt.show()

## 5. cos θ* distribution in the signal region

Main angular observable: **$\cos \theta^* = \left| \tanh\left(\frac{\Delta \eta}{2}\right) \right| \quad \text{with } \Delta \eta = \eta_{\text{jet0}} - \eta_{\text{jet1}}$** (two leading jets). Same SR selection as MET; MC-only, scaled to $41.5\,\mathrm{fb}^{-1}$.


In [ ]:
# cos(theta*) in SR (main observable): stack backgrounds + overlay two selected signal mass points
trigger_list = get_trigger_list()
bins_cts = np.linspace(0, 1, 5)
cts_arrays, cts_w, cts_labels, cts_keys = [], [], [], []
for name in bkg_names:
    if name not in events_by_dataset:
        continue
    norm = norm_factors.get(name)
    if norm is None:
        continue
    ev = events_by_dataset[name]
    hlt_fields = set(ev.HLT.fields) if hasattr(ev, "HLT") and hasattr(ev.HLT, "fields") else set()
    trigger_mask = ak.zeros_like(ev.event, dtype=bool)
    for tname in trigger_list:
        if tname in hlt_fields:
            trigger_mask = trigger_mask | ev.HLT[tname]
    ev = ev[trigger_mask]
    good_jets = select_good_jets(ev)
    njets = ak.num(good_jets)
    nbjets = count_bjets(good_jets)
    nlep = n_leptons(ev)
    met_phi = ev.MET.phi
    recoil_evt = get_recoil(ev)
    dphi = np.abs(good_jets.phi - met_phi)
    dphi = ak.where(dphi > np.pi, 2 * np.pi - dphi, dphi)
    min_dphi = ak.min(dphi, axis=1)
    dphi_cut = min_dphi > 0.5

    sr_mask = (
        (njets >= 2)
        & (njets <= 3)
        & (nbjets == 2)
        & (nlep == 0)
        & dphi_cut
        & met_pf_calo_mask(ev)
        & (recoil_evt > RECOIL_MIN)
    )

    good_jets_sr = good_jets[sr_mask]
    has_two = ak.num(good_jets_sr) >= 2
    jets_pad = ak.pad_none(good_jets_sr, 2)
    jet0 = jets_pad[:, 0]
    jet1 = jets_pad[:, 1]
    mask = has_two & ~ak.is_none(jet1)
    if ak.sum(mask) == 0:
        continue
    cts = cos_theta_star(jet0[mask], jet1[mask])
    flat = ak.to_numpy(ak.ravel(cts))
    if len(flat) == 0:
        continue
    cts_arrays.append(flat)
    cts_w.append(np.full_like(flat, norm, dtype=float))
    cts_labels.append(latex_labels.get(name, labels.get(name, name)))
    cts_keys.append(name)

fig, ax_main, ax_ratio = plot_stacked_with_ratio(
    arrs=cts_arrays,
    warrs=cts_w,
    labels=cts_labels,
    colors=[PROCESS_COLORS.get(k, DEFAULT_STACK_COLOR) for k in cts_keys],
    bins=bins_cts,
    xlabel="$\\cos(\\theta)*$",
    title="$\\cos(\\theta)$*",
    data_vals=None,
    yscale_log=True,
)

SIGNAL_VIS_SCALE = 1
sig_name = "bbDM_2HDMa_LO_5f"
sig_masspoints = [
    ("Signal mA=600, ma=250", "MH3_600_MH4_250_Mchi_1"),
    ("Signal mA=600, ma=500", "MH3_600_MH4_500_Mchi_1"),
]
if sig_name in events_by_dataset:
    ev0 = events_by_dataset[sig_name]
    hlt_fields = set(ev0.HLT.fields) if hasattr(ev0, "HLT") and hasattr(ev0.HLT, "fields") else set()
    trigger_mask = ak.zeros_like(ev0.event, dtype=bool)
    for tname in trigger_list:
        if tname in hlt_fields:
            trigger_mask = trigger_mask | ev0.HLT[tname]
    if ak.sum(trigger_mask) == 0:
        trigger_mask = ak.ones_like(ev0.event, dtype=bool)
    ev0 = ev0[trigger_mask]

    gm_fields = set(ev0.GenModel.fields) if hasattr(ev0, "GenModel") and hasattr(ev0.GenModel, "fields") else set()
    for sig_label, gm_field in sig_masspoints:
        if gm_field not in gm_fields:
            continue

        ev = ev0[ak.fill_none(ev0.GenModel[gm_field] != 0, False)]
        good_jets = select_good_jets(ev)
        njets = ak.num(good_jets)
        nbjets = count_bjets(good_jets)
        nlep = n_leptons(ev)
        met_phi = ev.MET.phi
        recoil_evt = get_recoil(ev)
        dphi = np.abs(good_jets.phi - met_phi)
        dphi = ak.where(dphi > np.pi, 2 * np.pi - dphi, dphi)
        min_dphi = ak.min(dphi, axis=1)
        dphi_cut = min_dphi > 0.5
        sr_base = (
            (njets >= 2)
            & (njets <= 3)
            & (nbjets == 2)
            & (nlep == 0)
            & dphi_cut
            & met_pf_calo_mask(ev)
            & (recoil_evt > RECOIL_MIN)
        )
        gj = good_jets[sr_base]
        has_two = ak.num(gj) >= 2
        jp = ak.pad_none(gj, 2)
        j0 = jp[:, 0]
        j1 = jp[:, 1]
        valid = has_two & ~ak.is_none(j1)
        if ak.sum(valid) > 0:
            sval = ak.to_numpy(ak.ravel(cos_theta_star(j0[valid], j1[valid])))
            if len(sval) > 0:
                sw = np.full_like(sval, SIGNAL_VIS_SCALE, dtype=float)
                ax_main.hist(sval, bins=bins_cts, weights=sw, histtype="step", linewidth=2.0, linestyle="--", label=f"{sig_label} (x{SIGNAL_VIS_SCALE:g})")
ax_main.legend(loc="upper right", ncol=2, fontsize=9, frameon=False, columnspacing=1.0, handlelength=1.6)
plt.show()

## 7. Z control regions (zecr and zmucr) — data and MC

Preselection: 
- $\geq 1$ jet, leading jet $p_{\mathrm{T}} > 100~\mathrm{GeV}$

- $\Delta\phi(\text{jet}, \mathrm{MET}) > 0.5$

- $\mathrm{PF\text{-}calo~MET} < 0.5~\mathrm{GeV}$

- Recoil $> 250~\mathrm{GeV}$  
    - $\left(U = -\left(\vec{p}_{\mathrm{T}}^{\,\text{miss}} + \sum \vec{p}_{\mathrm{T}}^{\,\ell}\right)\right)$

- Exactly 2 OSSF leptons (ee or $\mu\mu$)

- Leading lepton $p_{\mathrm{T}} > 30~\mathrm{GeV}$

- $70 < m_{\ell\ell} < 110~\mathrm{GeV}$

In [ ]:
from processor.bbdm_processor import select_tight_electrons, select_tight_muons, recoil_pt, met_pf_calo_mask

PRESEL_RECOIL_MIN = 250.0
LEAD_JET_PT_MIN = 100.0
Z_CR_MLL_LO, Z_CR_MLL_HI = 70.0, 110.0
LEAD_LEP_PT_CR = 30.0

def z_cr_components(events):
    good_jets = select_good_jets(events)
    njets = ak.num(good_jets)
    lead_jet_pt = ak.fill_none(ak.firsts(good_jets.pt), 0.0)
    met_pt, met_phi = events.MET.pt, events.MET.phi
    tight_ele = select_tight_electrons(events)
    tight_mu = select_tight_muons(events)
    nele, nmu = ak.count(tight_ele.pt, axis=1), ak.count(tight_mu.pt, axis=1)
    two_ee = (nele == 2) & (nmu == 0) & (ak.sum(tight_ele.charge, axis=1) == 0)
    two_mumu = (nele == 0) & (nmu == 2) & (ak.sum(tight_mu.charge, axis=1) == 0)

    tight_ele_pad = ak.pad_none(tight_ele, 2)
    tight_mu_pad = ak.pad_none(tight_mu, 2)
    pair_ee = tight_ele_pad[:, 0] + tight_ele_pad[:, 1]
    pair_mumu = tight_mu_pad[:, 0] + tight_mu_pad[:, 1]

    sum_lep_px = ak.where(two_ee, ak.fill_none(pair_ee.pt, 0.0) * np.cos(ak.fill_none(pair_ee.phi, 0.0)), ak.where(two_mumu, ak.fill_none(pair_mumu.pt, 0.0) * np.cos(ak.fill_none(pair_mumu.phi, 0.0)), ak.full_like(met_pt, 0.0)))
    sum_lep_py = ak.where(two_ee, ak.fill_none(pair_ee.pt, 0.0) * np.sin(ak.fill_none(pair_ee.phi, 0.0)), ak.where(two_mumu, ak.fill_none(pair_mumu.pt, 0.0) * np.sin(ak.fill_none(pair_mumu.phi, 0.0)), ak.full_like(met_pt, 0.0)))

    recoil = recoil_pt(met_pt, met_phi, sum_lep_px, sum_lep_py)

    dphi_j = np.abs(good_jets.phi - met_phi)
    dphi_j = ak.where(dphi_j > np.pi, 2 * np.pi - dphi_j, dphi_j)
    min_dphi = ak.min(dphi_j, axis=1)
    dphi_cut = min_dphi > 0.5
    met_pf_calo_ok = met_pf_calo_mask(events, mode="cr", sum_lep_px=sum_lep_px, sum_lep_py=sum_lep_py)

    presel = (njets >= 1) & (lead_jet_pt > LEAD_JET_PT_MIN) & dphi_cut & met_pf_calo_ok & (recoil > PRESEL_RECOIL_MIN)
    mll_ee = ak.where(two_ee, ak.fill_none(pair_ee.mass, -1.0), ak.full_like(met_pt, -1.0))
    mll_mumu = ak.where(two_mumu, ak.fill_none(pair_mumu.mass, -1.0), ak.full_like(met_pt, -1.0))
    lead_lep_pt = ak.where(two_ee, ak.max(tight_ele.pt, axis=1), ak.where(two_mumu, ak.max(tight_mu.pt, axis=1), ak.full_like(met_pt, 0.0)))

    zecr = presel & two_ee & (lead_lep_pt > LEAD_LEP_PT_CR) & (mll_ee > Z_CR_MLL_LO) & (mll_ee < Z_CR_MLL_HI)
    zmucr = presel & two_mumu & (lead_lep_pt > LEAD_LEP_PT_CR) & (mll_mumu > Z_CR_MLL_LO) & (mll_mumu < Z_CR_MLL_HI)
    return zecr, zmucr, recoil, good_jets


In [ ]:
# zecr recoil: stacked backgrounds (SR style, no signal overlay)
bins_recoil = np.array([250, 300, 400, 550, 1000], dtype=float)
arrs, warrs, labs, stack_keys = [], [], [], []
for name in bkg_names:
    if name not in events_by_dataset:
        continue
    norm = norm_factors.get(name)
    if norm is None:
        continue
    zecr, _, recoil, _ = z_cr_components(events_by_dataset[name])
    vals = ak.to_numpy(ak.ravel(recoil[zecr]))
    if len(vals) == 0:
        continue
    arrs.append(vals)
    warrs.append(np.full_like(vals, norm, dtype=float))
    labs.append(latex_labels.get(name, labels.get(name, name)))
    stack_keys.append(name)

data_vals_list = []
for dname, d_ev in events_by_dataset.items():
    if (not is_data.get(dname, False)) or (not str(dname).startswith("SingleElectron")):
        continue
    zecr_d, _, recoil_d, _ = z_cr_components(d_ev)
    vals_d = ak.to_numpy(ak.ravel(recoil_d[zecr_d]))
    if len(vals_d) > 0:
        data_vals_list.append(vals_d)
data_vals = np.concatenate(data_vals_list) if data_vals_list else None
plot_stacked_with_ratio(
    arrs=arrs,
    warrs=warrs,
    labels=labs,
    colors=[PROCESS_COLORS.get(k, DEFAULT_STACK_COLOR) for k in stack_keys],
    bins=bins_recoil,
    xlabel="Recoil [GeV]",
    title="zecr (ee): Recoil",
    data_vals=data_vals,
    yscale_log=True,
)


In [ ]:
# zmucr recoil: stacked backgrounds (SR style, no signal overlay)
bins_recoil = np.array([250, 300, 400, 550, 1000], dtype=float)
arrs, warrs, labs, stack_keys = [], [], [], []
for name in bkg_names:
    if name not in events_by_dataset:
        continue
    norm = norm_factors.get(name)
    if norm is None:
        continue
    _, zmucr, recoil, _ = z_cr_components(events_by_dataset[name])
    vals = ak.to_numpy(ak.ravel(recoil[zmucr]))
    if len(vals) == 0:
        continue
    arrs.append(vals)
    warrs.append(np.full_like(vals, norm, dtype=float))
    labs.append(latex_labels.get(name, labels.get(name, name)))
    stack_keys.append(name)

data_vals_list = []
for dname, d_ev in events_by_dataset.items():
    if (not is_data.get(dname, False)) or (not str(dname).startswith("MET")):
        continue
    _, zmucr_d, recoil_d, _ = z_cr_components(d_ev)
    vals_d = ak.to_numpy(ak.ravel(recoil_d[zmucr_d]))
    if len(vals_d) > 0:
        data_vals_list.append(vals_d)
data_vals = np.concatenate(data_vals_list) if data_vals_list else None
plot_stacked_with_ratio(
    arrs=arrs,
    warrs=warrs,
    labels=labs,
    colors=[PROCESS_COLORS.get(k, DEFAULT_STACK_COLOR) for k in stack_keys],
    bins=bins_recoil,
    xlabel="Recoil [GeV]",
    title="zmucr (mumu): Recoil",
    data_vals=data_vals,
    yscale_log=True,
)


In [ ]:
# zecr dilepton mass (m_ll): stacked backgrounds (no signal overlay)
bins_mll = np.linspace(70, 110, 21)
arrs, warrs, labs, stack_keys = [], [], [], []
for name in bkg_names:
    if name not in events_by_dataset:
        continue
    norm = norm_factors.get(name)
    if norm is None:
        continue
    ev = events_by_dataset[name]
    zecr, _, _, _ = z_cr_components(ev)
    tight_ele = select_tight_electrons(ev)
    pair_ee = ak.pad_none(tight_ele, 2)[:, 0] + ak.pad_none(tight_ele, 2)[:, 1]
    vals = ak.to_numpy(ak.ravel(ak.fill_none(pair_ee.mass[zecr], -1.0)))
    vals = vals[vals > 0]
    if len(vals) == 0:
        continue
    arrs.append(vals)
    warrs.append(np.full_like(vals, norm, dtype=float))
    labs.append(latex_labels.get(name, labels.get(name, name)))
    stack_keys.append(name)

data_vals_list = []
for dname, d_ev in events_by_dataset.items():
    if (not is_data.get(dname, False)) or (not str(dname).startswith("SingleElectron")):
        continue
    zecr_d, _, _, _ = z_cr_components(d_ev)
    tight_ele_d = select_tight_electrons(d_ev)
    pair_ee_d = ak.pad_none(tight_ele_d, 2)[:, 0] + ak.pad_none(tight_ele_d, 2)[:, 1]
    vals_d = ak.to_numpy(ak.ravel(ak.fill_none(pair_ee_d.mass[zecr_d], -1.0)))
    vals_d = vals_d[vals_d > 0]
    if len(vals_d) > 0:
        data_vals_list.append(vals_d)
data_vals = np.concatenate(data_vals_list) if data_vals_list else None
plot_stacked_with_ratio(
    arrs=arrs,
    warrs=warrs,
    labels=labs,
    colors=[PROCESS_COLORS.get(k, DEFAULT_STACK_COLOR) for k in stack_keys],
    bins=bins_mll,
    xlabel="$m_{ee}$ [GeV]",
    title="zecr (ee): dilepton mass",
    data_vals=data_vals,
    yscale_log=False,
)


In [ ]:
# zmucr dilepton mass (m_ll): stacked backgrounds (no signal overlay)
bins_mll = np.linspace(70, 110, 21)
arrs, warrs, labs, stack_keys = [], [], [], []
for name in bkg_names:
    if name not in events_by_dataset:
        continue
    norm = norm_factors.get(name)
    if norm is None:
        continue
    ev = events_by_dataset[name]
    _, zmucr, _, _ = z_cr_components(ev)
    tight_mu = select_tight_muons(ev)
    pair_mumu = ak.pad_none(tight_mu, 2)[:, 0] + ak.pad_none(tight_mu, 2)[:, 1]
    vals = ak.to_numpy(ak.ravel(ak.fill_none(pair_mumu.mass[zmucr], -1.0)))
    vals = vals[vals > 0]
    if len(vals) == 0:
        continue
    arrs.append(vals)
    warrs.append(np.full_like(vals, norm, dtype=float))
    labs.append(latex_labels.get(name, labels.get(name, name)))
    stack_keys.append(name)

data_vals_list = []
for dname, d_ev in events_by_dataset.items():
    if (not is_data.get(dname, False)) or (not str(dname).startswith("MET")):
        continue
    _, zmucr_d, _, _ = z_cr_components(d_ev)
    tight_mu_d = select_tight_muons(d_ev)
    pair_mumu_d = ak.pad_none(tight_mu_d, 2)[:, 0] + ak.pad_none(tight_mu_d, 2)[:, 1]
    vals_d = ak.to_numpy(ak.ravel(ak.fill_none(pair_mumu_d.mass[zmucr_d], -1.0)))
    vals_d = vals_d[vals_d > 0]
    if len(vals_d) > 0:
        data_vals_list.append(vals_d)
data_vals = np.concatenate(data_vals_list) if data_vals_list else None
plot_stacked_with_ratio(
    arrs=arrs,
    warrs=warrs,
    labels=labs,
    colors=[PROCESS_COLORS.get(k, DEFAULT_STACK_COLOR) for k in stack_keys],
    bins=bins_mll,
    xlabel="$m_{\mu\mu}$ [GeV]",
    title="zmucr (mumu): dilepton mass",
    data_vals=data_vals,
    yscale_log=False,
)


In [ ]:
# zecr cos(theta*): stacked backgrounds (SR style, no signal overlay)
bins_cts = np.linspace(0, 1, 5)
arrs, warrs, labs, stack_keys = [], [], [], []
for name in bkg_names:
    if name not in events_by_dataset:
        continue
    norm = norm_factors.get(name)
    if norm is None:
        continue
    zecr, _, _, good_jets = z_cr_components(events_by_dataset[name])
    gj = good_jets[zecr]
    has_two = ak.num(gj) >= 2
    jp = ak.pad_none(gj, 2)
    j0, j1 = jp[:, 0], jp[:, 1]
    valid = has_two & ~ak.is_none(j1)
    if ak.sum(valid) == 0:
        continue
    vals = ak.to_numpy(ak.ravel(cos_theta_star(j0[valid], j1[valid])))
    arrs.append(vals)
    warrs.append(np.full_like(vals, norm, dtype=float))
    labs.append(latex_labels.get(name, labels.get(name, name)))
    stack_keys.append(name)

data_vals_list = []
for dname, d_ev in events_by_dataset.items():
    if (not is_data.get(dname, False)) or (not str(dname).startswith("SingleElectron")):
        continue
    zecr_d, _, _, good_jets_d = z_cr_components(d_ev)
    gj_d = good_jets_d[zecr_d]
    has_two_d = ak.num(gj_d) >= 2
    jp_d = ak.pad_none(gj_d, 2)
    j0_d, j1_d = jp_d[:, 0], jp_d[:, 1]
    valid_d = has_two_d & ~ak.is_none(j1_d)
    if ak.sum(valid_d) == 0:
        continue
    vals_d = ak.to_numpy(ak.ravel(cos_theta_star(j0_d[valid_d], j1_d[valid_d])))
    if len(vals_d) > 0:
        data_vals_list.append(vals_d)
data_vals = np.concatenate(data_vals_list) if data_vals_list else None
plot_stacked_with_ratio(
    arrs=arrs,
    warrs=warrs,
    labels=labs,
    colors=[PROCESS_COLORS.get(k, DEFAULT_STACK_COLOR) for k in stack_keys],
    bins=bins_cts,
    xlabel="$\\cos(\\theta)*$",
    title="zecr (ee): cos",
    data_vals=data_vals,
    yscale_log=True,
)


In [ ]:
# zmucr cos(theta*): stacked backgrounds (SR style, no signal overlay)
bins_cts = np.linspace(0, 1, 5)
arrs, warrs, labs, stack_keys = [], [], [], []
for name in bkg_names:
    if name not in events_by_dataset:
        continue
    norm = norm_factors.get(name)
    if norm is None:
        continue
    _, zmucr, _, good_jets = z_cr_components(events_by_dataset[name])
    gj = good_jets[zmucr]
    has_two = ak.num(gj) >= 2
    jp = ak.pad_none(gj, 2)
    j0, j1 = jp[:, 0], jp[:, 1]
    valid = has_two & ~ak.is_none(j1)
    if ak.sum(valid) == 0:
        continue
    vals = ak.to_numpy(ak.ravel(cos_theta_star(j0[valid], j1[valid])))
    arrs.append(vals)
    warrs.append(np.full_like(vals, norm, dtype=float))
    labs.append(latex_labels.get(name, labels.get(name, name)))
    stack_keys.append(name)

data_vals_list = []
for dname, d_ev in events_by_dataset.items():
    if (not is_data.get(dname, False)) or (not str(dname).startswith("MET")):
        continue
    _, zmucr_d, _, good_jets_d = z_cr_components(d_ev)
    gj_d = good_jets_d[zmucr_d]
    has_two_d = ak.num(gj_d) >= 2
    jp_d = ak.pad_none(gj_d, 2)
    j0_d, j1_d = jp_d[:, 0], jp_d[:, 1]
    valid_d = has_two_d & ~ak.is_none(j1_d)
    if ak.sum(valid_d) == 0:
        continue
    vals_d = ak.to_numpy(ak.ravel(cos_theta_star(j0_d[valid_d], j1_d[valid_d])))
    if len(vals_d) > 0:
        data_vals_list.append(vals_d)
data_vals = np.concatenate(data_vals_list) if data_vals_list else None
plot_stacked_with_ratio(
    arrs=arrs,
    warrs=warrs,
    labels=labs,
    colors=[PROCESS_COLORS.get(k, DEFAULT_STACK_COLOR) for k in stack_keys],
    bins=bins_cts,
    xlabel="$\\cos(\\theta)*$",
    title="zmucr (mumu): cos",
    data_vals=data_vals,
    yscale_log=True,
)


## 7. Top control regions (tecr and tmucr) — data and MC

- Exactly one lepton with $p_{\mathrm{T}} > 30~\mathrm{GeV}$

- $m_{\mathrm{T}} < 160~\mathrm{GeV}$

- $\geq 2$ b-jets

- $\geq 2$ non-b jets

In [ ]:
from processor.bbdm_processor import met_pf_calo_mask

TOP_CR_MT_MAX = 160.0

def top_cr_components(events):
    good_jets = select_good_jets(events)
    njets = ak.num(good_jets)
    nbjets = count_bjets(good_jets)
    lead_jet_pt = ak.fill_none(ak.firsts(good_jets.pt), 0.0)
    met_pt, met_phi = events.MET.pt, events.MET.phi
    tight_ele = select_tight_electrons(events)
    tight_mu = select_tight_muons(events)

    one_ele = (ak.count(tight_ele.pt, axis=1) == 1) & (ak.count(tight_mu.pt, axis=1) == 0)
    one_mu = (ak.count(tight_ele.pt, axis=1) == 0) & (ak.count(tight_mu.pt, axis=1) == 1)

    lep_pt = ak.fill_none(ak.where(one_ele, ak.firsts(tight_ele.pt), ak.where(one_mu, ak.firsts(tight_mu.pt), ak.full_like(met_pt, 0.0))), 0.0)
    lep_phi = ak.fill_none(ak.where(one_ele, ak.firsts(tight_ele.phi), ak.where(one_mu, ak.firsts(tight_mu.phi), ak.full_like(met_pt, 0.0))), 0.0)
    sum_lep_px = lep_pt * np.cos(lep_phi)
    sum_lep_py = lep_pt * np.sin(lep_phi)

    recoil = recoil_pt(met_pt, met_phi, sum_lep_px, sum_lep_py)

    dphi_j = np.abs(good_jets.phi - met_phi)
    dphi_j = ak.where(dphi_j > np.pi, 2 * np.pi - dphi_j, dphi_j)
    min_dphi = ak.min(dphi_j, axis=1)
    dphi_cut = min_dphi > 0.5
    met_pf_calo_ok = met_pf_calo_mask(events, mode="cr", sum_lep_px=sum_lep_px, sum_lep_py=sum_lep_py)
    presel = (njets >= 1) & (lead_jet_pt > LEAD_JET_PT_MIN) & dphi_cut & met_pf_calo_ok & (recoil > PRESEL_RECOIL_MIN)

    dphi = np.abs(ak.to_numpy(met_phi) - ak.to_numpy(lep_phi))
    dphi = np.where(dphi > np.pi, 2 * np.pi - dphi, dphi)
    mt = ak.Array(np.sqrt(2.0 * ak.to_numpy(met_pt) * ak.to_numpy(lep_pt) * (1.0 - np.cos(dphi))))

    n_non_b = njets - nbjets
    common = presel & (lep_pt > LEAD_LEP_PT_CR) & (mt < TOP_CR_MT_MAX) & (nbjets >= 2) & (n_non_b >= 2)
    tecr = common & one_ele
    tmucr = common & one_mu
    return tecr, tmucr, recoil, good_jets


In [ ]:
# tecr recoil: stacked backgrounds (SR style, no signal overlay)
bins_recoil = np.array([250, 300, 400, 550, 1000], dtype=float)
arrs, warrs, labs, stack_keys = [], [], [], []
for name in bkg_names:
    if name not in events_by_dataset:
        continue
    norm = norm_factors.get(name)
    if norm is None:
        continue
    tecr, _, recoil, _ = top_cr_components(events_by_dataset[name])
    vals = ak.to_numpy(ak.ravel(recoil[tecr]))
    if len(vals) == 0:
        continue
    arrs.append(vals)
    warrs.append(np.full_like(vals, norm, dtype=float))
    labs.append(latex_labels.get(name, labels.get(name, name)))
    stack_keys.append(name)

data_vals_list = []
for dname, d_ev in events_by_dataset.items():
    if (not is_data.get(dname, False)) or (not str(dname).startswith("SingleElectron")):
        continue
    tecr_d, _, recoil_d, _ = top_cr_components(d_ev)
    vals_d = ak.to_numpy(ak.ravel(recoil_d[tecr_d]))
    if len(vals_d) > 0:
        data_vals_list.append(vals_d)
data_vals = np.concatenate(data_vals_list) if data_vals_list else None
plot_stacked_with_ratio(
    arrs=arrs,
    warrs=warrs,
    labels=labs,
    colors=[PROCESS_COLORS.get(k, DEFAULT_STACK_COLOR) for k in stack_keys],
    bins=bins_recoil,
    xlabel="Recoil [GeV]",
    title="tecr (electron): Recoil",
    data_vals=data_vals,
    yscale_log=True,
)


In [ ]:
# tmucr recoil: stacked backgrounds (SR style, no signal overlay)
bins_recoil = np.array([250, 300, 400, 550, 1000], dtype=float)
arrs, warrs, labs, stack_keys = [], [], [], []
for name in bkg_names:
    if name not in events_by_dataset:
        continue
    norm = norm_factors.get(name)
    if norm is None:
        continue
    _, tmucr, recoil, _ = top_cr_components(events_by_dataset[name])
    vals = ak.to_numpy(ak.ravel(recoil[tmucr]))
    if len(vals) == 0:
        continue
    arrs.append(vals)
    warrs.append(np.full_like(vals, norm, dtype=float))
    labs.append(latex_labels.get(name, labels.get(name, name)))
    stack_keys.append(name)

data_vals_list = []
for dname, d_ev in events_by_dataset.items():
    if (not is_data.get(dname, False)) or (not str(dname).startswith("MET")):
        continue
    _, tmucr_d, recoil_d, _ = top_cr_components(d_ev)
    vals_d = ak.to_numpy(ak.ravel(recoil_d[tmucr_d]))
    if len(vals_d) > 0:
        data_vals_list.append(vals_d)
data_vals = np.concatenate(data_vals_list) if data_vals_list else None
plot_stacked_with_ratio(
    arrs=arrs,
    warrs=warrs,
    labels=labs,
    colors=[PROCESS_COLORS.get(k, DEFAULT_STACK_COLOR) for k in stack_keys],
    bins=bins_recoil,
    xlabel="Recoil [GeV]",
    title="tmucr (muon): Recoil",
    data_vals=data_vals,
    yscale_log=True,
)


In [ ]:
# tecr cos(theta*): stacked backgrounds (SR style, no signal overlay)
bins_cts = np.linspace(0, 1, 5)
arrs, warrs, labs, stack_keys = [], [], [], []
for name in bkg_names:
    if name not in events_by_dataset:
        continue
    norm = norm_factors.get(name)
    if norm is None:
        continue
    tecr, _, _, good_jets = top_cr_components(events_by_dataset[name])
    gj = good_jets[tecr]
    has_two = ak.num(gj) >= 2
    jp = ak.pad_none(gj, 2)
    j0, j1 = jp[:, 0], jp[:, 1]
    valid = has_two & ~ak.is_none(j1)
    if ak.sum(valid) == 0:
        continue
    vals = ak.to_numpy(ak.ravel(cos_theta_star(j0[valid], j1[valid])))
    arrs.append(vals)
    warrs.append(np.full_like(vals, norm, dtype=float))
    labs.append(latex_labels.get(name, labels.get(name, name)))
    stack_keys.append(name)

data_vals_list = []
for dname, d_ev in events_by_dataset.items():
    if (not is_data.get(dname, False)) or (not str(dname).startswith("SingleElectron")):
        continue
    tecr_d, _, _, good_jets_d = top_cr_components(d_ev)
    gj_d = good_jets_d[tecr_d]
    has_two_d = ak.num(gj_d) >= 2
    jp_d = ak.pad_none(gj_d, 2)
    j0_d, j1_d = jp_d[:, 0], jp_d[:, 1]
    valid_d = has_two_d & ~ak.is_none(j1_d)
    if ak.sum(valid_d) == 0:
        continue
    vals_d = ak.to_numpy(ak.ravel(cos_theta_star(j0_d[valid_d], j1_d[valid_d])))
    if len(vals_d) > 0:
        data_vals_list.append(vals_d)
data_vals = np.concatenate(data_vals_list) if data_vals_list else None
plot_stacked_with_ratio(
    arrs=arrs,
    warrs=warrs,
    labels=labs,
    colors=[PROCESS_COLORS.get(k, DEFAULT_STACK_COLOR) for k in stack_keys],
    bins=bins_cts,
    xlabel="$\\cos(\\theta)*$",
    title="tecr (electron): $\\cos(\\theta)*$",
    data_vals=data_vals,
    yscale_log=True,
)


In [ ]:
# tmucr cos(theta*): stacked backgrounds (SR style, no signal overlay)
bins_cts = np.linspace(0, 1, 5)
arrs, warrs, labs, stack_keys = [], [], [], []
for name in bkg_names:
    if name not in events_by_dataset:
        continue
    norm = norm_factors.get(name)
    if norm is None:
        continue
    _, tmucr, _, good_jets = top_cr_components(events_by_dataset[name])
    gj = good_jets[tmucr]
    has_two = ak.num(gj) >= 2
    jp = ak.pad_none(gj, 2)
    j0, j1 = jp[:, 0], jp[:, 1]
    valid = has_two & ~ak.is_none(j1)
    if ak.sum(valid) == 0:
        continue
    vals = ak.to_numpy(ak.ravel(cos_theta_star(j0[valid], j1[valid])))
    arrs.append(vals)
    warrs.append(np.full_like(vals, norm, dtype=float))
    labs.append(latex_labels.get(name, labels.get(name, name)))
    stack_keys.append(name)

data_vals_list = []
for dname, d_ev in events_by_dataset.items():
    if (not is_data.get(dname, False)) or (not str(dname).startswith("MET")):
        continue
    _, tmucr_d, _, good_jets_d = top_cr_components(d_ev)
    gj_d = good_jets_d[tmucr_d]
    has_two_d = ak.num(gj_d) >= 2
    jp_d = ak.pad_none(gj_d, 2)
    j0_d, j1_d = jp_d[:, 0], jp_d[:, 1]
    valid_d = has_two_d & ~ak.is_none(j1_d)
    if ak.sum(valid_d) == 0:
        continue
    vals_d = ak.to_numpy(ak.ravel(cos_theta_star(j0_d[valid_d], j1_d[valid_d])))
    if len(vals_d) > 0:
        data_vals_list.append(vals_d)
data_vals = np.concatenate(data_vals_list) if data_vals_list else None
plot_stacked_with_ratio(
    arrs=arrs,
    warrs=warrs,
    labels=labs,
    colors=[PROCESS_COLORS.get(k, DEFAULT_STACK_COLOR) for k in stack_keys],
    bins=bins_cts,
    xlabel="$\\cos(\\theta)*$",
    title="tmucr (muon): $\\cos(\\theta)*$",
    data_vals=data_vals,
    yscale_log=True,
)


In [ ]:
# Region-wise cutflow / yields after trigger (SR, zecr, zmucr, tecr, tmucr)
print("Region yields by dataset:")
for name, ev0 in events_by_dataset.items():
    hlt_fields = set(ev0.HLT.fields) if hasattr(ev0, "HLT") and hasattr(ev0.HLT, "fields") else set()
    trigger_mask = ak.zeros_like(ev0.event, dtype=bool)
    for tname in get_trigger_list():
        if tname in hlt_fields:
            trigger_mask = trigger_mask | ev0.HLT[tname]
    ev = ev0[trigger_mask]

    sr_counts = run_cutflow(ev)
    recoil_key = f"Recoil>{int(RECOIL_MIN)}"
    zecr, zmucr, _, _ = z_cr_components(ev)
    tecr, tmucr, _, _ = top_cr_components(ev)

    print(labels.get(name, name), {
        "trigger": int(len(ev)),
        "sr_final": int(sr_counts[recoil_key]),
        "zecr": int(ak.sum(zecr)),
        "zmucr": int(ak.sum(zmucr)),
        "tecr": int(ak.sum(tecr)),
        "tmucr": int(ak.sum(tmucr)),
    })

## 9. Comparison plots (Ex 3.4)

In [ ]:
# Use first MC background for comparison plot
name = bkg_names[0] if bkg_names else list(events_by_dataset.keys())[0]
ev = events_by_dataset[name]
good_jets = select_good_jets(ev)
njets = ak.num(good_jets)
nbjets = count_bjets(good_jets)
nlep = n_leptons(ev)
recoil_evt = get_recoil(ev)

sr_mask = (njets >= 2) & (njets <= 3) & (nbjets == 2) & (nlep == 0) & (recoil_evt > RECOIL_MIN)
recoil_presel = recoil_evt[(njets >= 2) & (njets <= 3) & (nbjets == 2)]
recoil_sr = recoil_evt[sr_mask]

fig, ax = plt.subplots(1, 2, figsize=(15, 8))
ax[0].hist(ak.to_numpy(ak.ravel(recoil_presel)), bins=50, range=(0, 500), edgecolor="black", alpha=0.7, label="Presel")
ax[0].set_xlabel("Recoil [GeV]")
ax[0].set_ylabel("Events")
ax[0].set_title("Recoil (preselection)")
ax[1].hist(ak.to_numpy(ak.ravel(recoil_sr)), bins=50, range=(200, 600), edgecolor="black", alpha=0.7, label="SR")
ax[1].set_xlabel("Recoil [GeV]")
ax[1].set_ylabel("Events")
ax[1].set_title("Recoil (signal region)")
plt.tight_layout()
plt.show()


## Save Short Analysis Output in Full-Analysis Structure

The next cell processes the loaded `events_by_dataset` with `bbDMProcessor`, applies MC normalization (`xsec*lumi/Ngen`), merges samples into analysis groups, and saves a structured bundle to:

`output/output_2017_short.pkl`

with top-level keys:

- `metadata`
- `samples`

In [ ]:
from pathlib import Path
import pickle

from processor.bbdm_processor import bbDMProcessor
from config.datasets_2017 import load_merge_config, merge_processor_results_by_group


def _project_root_from_cwd() -> Path:
    cwd = Path.cwd()
    if (cwd / "config").exists() and (cwd / "output").exists():
        return cwd
    if (cwd.parent / "config").exists() and (cwd.parent / "output").exists():
        return cwd.parent
    raise FileNotFoundError("Could not locate project root (needs config/ and output/).")


def _scale_histograms_in_accumulator(acc, factor: float) -> None:
    for value in acc.values():
        if hasattr(value, "_hist"):
            value._hist = value._hist * float(factor)


def _schema_sample_key(name: str) -> str:
    if name == "MET":
        return "data_MET"
    if name == "SingleElectron":
        return "data_SingleElectron"
    return str(name)


def _accumulator_to_schema_sample(acc) -> dict:
    sample = {"cutflow": dict(acc.get("cutflow", {}))}
    hists_by_region = {}
    hists = {}
    for key, value in acc.items():
        if key == "cutflow":
            continue
        if hasattr(value, "_hist"):
            if str(key).endswith("_by_region"):
                hists_by_region[str(key)[:-10]] = value
            else:
                hists[str(key)] = value
    sample["hists_by_region"] = hists_by_region
    if hists:
        sample["hists"] = hists
    return sample


# 1) Process each loaded dataset with the same processor used in full analysis.
proc = bbDMProcessor()
per_dataset_results = {}
for name, ev in events_by_dataset.items():
    out = proc.process(ev)

    # Apply MC normalization (xsec*lumi/Ngen), data stays unscaled.
    if not is_data.get(name, False):
        xsec = xsecs.get(name)
        n_gen = float(out.get("cutflow", {}).get("all_events", 0))
        if xsec is not None and n_gen > 0.0:
            sf = (float(xsec) * LUMI_PB) / n_gen
            _scale_histograms_in_accumulator(out, sf)
    per_dataset_results[name] = out

# 2) Merge into physics groups, while splitting data streams as MET/SingleElectron.
merge_map, prefix_rules = load_merge_config()
for name in list(per_dataset_results.keys()):
    n = str(name)
    if n.startswith("MET_Run2017") or n.startswith("SingleMuon_Run2017") or n.startswith("SingleMuon"):
        merge_map[n] = "MET"
    elif n.startswith("SingleElectron_Run2017") or n.startswith("SingleElectron"):
        merge_map[n] = "SingleElectron"

merged_results = merge_processor_results_by_group(
    per_dataset_results,
    merge_map=merge_map,
    prefix_rules=prefix_rules,
)

# 3) Build full-analysis style bundle and save to output/output_2017_short.pkl.
bundle = {
    "metadata": {
        "year": 2017,
        "lumi_fb": 41.5,
        "normalized": True,
        "normalization": "xsec*lumi/Ngen (MC), data=1",
        "regions": ["sr", "zecr", "zmucr", "tecr", "tmucr"],
        "data_stream_by_region": {
            "sr": "MET",
            "zmucr": "MET",
            "tmucr": "MET",
            "zecr": "SingleElectron",
            "tecr": "SingleElectron",
        },
        "binning": {
            "recoil": [250, 300, 400, 550, 1000],
            "cos_theta_star": [0.0, 0.25, 0.5, 0.75, 1.0],
        },
    },
    "samples": {
        _schema_sample_key(name): _accumulator_to_schema_sample(acc)
        for name, acc in merged_results.items()
    },
}

project_root = _project_root_from_cwd()
out_path = project_root / "output" / "output_2017_short.pkl"
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "wb") as f:
    pickle.dump(bundle, f)

print(f"Saved: {out_path}")
print("Sample keys:", sorted(bundle["samples"].keys()))

In [ ]:
from pathlib import Path
import pickle


def print_bundle_structure(pkl_path: str) -> None:
    with open(pkl_path, "rb") as f:
        bundle = pickle.load(f)

    print("Top-level keys:", list(bundle.keys()))
    metadata = bundle.get("metadata", {})
    samples = bundle.get("samples", {})

    print("\nmetadata")
    for k, v in metadata.items():
        print(f"  {k}: {v}")

    print(f"\nsamples: {len(samples)} entries")
    for sname in sorted(samples.keys()):
        s = samples[sname]
        hbr = s.get("hists_by_region", {}) if isinstance(s, dict) else {}
        h_other = s.get("hists", {}) if isinstance(s, dict) else {}
        cf = s.get("cutflow", {}) if isinstance(s, dict) else {}
        print(f"- {sname}")
        print(f"    cutflow_keys: {len(cf)}")
        print(f"    hists_by_region: {sorted(hbr.keys())}")
        if h_other:
            print(f"    hists: {sorted(h_other.keys())}")


cwd = Path.cwd()
if (cwd / "output" / "output_2017_short.pkl").is_file():
    input_pkl = cwd / "output" / "output_2017_short.pkl"
elif (cwd.parent / "output" / "output_2017_short.pkl").is_file():
    input_pkl = cwd.parent / "output" / "output_2017_short.pkl"
else:
    raise FileNotFoundError("output/output_2017_short.pkl not found. Run the previous save cell first.")

print_bundle_structure(str(input_pkl))

## Plot All Variables from the Full Output

Run the cell below to render all variables from the normalized full-analysis pickle:

In [ ]:
import importlib
from pathlib import Path

import config.plot_pkl_variables as pp
pp = importlib.reload(pp)

cwd = Path.cwd()
if (cwd / "output").exists() and (cwd / "config").exists():
    project_root = cwd
elif (cwd.parent / "output").exists() and (cwd.parent / "config").exists():
    project_root = cwd.parent
else:
    raise FileNotFoundError("Could not locate project root containing both 'config' and 'output' directories.")

pkl_path = project_root / "output" / "output_2017_full.pkl"
pp.plot_all_variables_grid(str(pkl_path))